<div style="background: linear-gradient(120deg, #1a3a5c 0%, #2d6a9f 60%, #4a9eda 100%); padding: 28px 36px; border-radius: 14px; display: flex; align-items: center; gap: 28px; box-shadow: 0 4px 18px rgba(0,0,0,0.18);">
    <img src='Figures/iteso.jpg' style="height: 110px; border-radius: 8px; background: white; padding: 6px; flex-shrink: 0; box-shadow: 0 2px 8px rgba(0,0,0,0.2);"/>
    <div style="border-left: 2px solid rgba(255,255,255,0.4); padding-left: 28px;">
        <h1 style="margin: 0 0 8px 0; color: white; font-size: 1.5em; line-height: 1.3;">Ingeniería y Ciencia de Datos</h1>
        <h3 style="margin: 0 0 8px 0; color: white; font-size: 1.5em; line-height: 1.3;">Laboratorio de Procesamiento de Datos</h3>
        <h3 style="margin: 0; color: rgba(255,255,255,0.8); font-weight: normal; font-size: 1.05em;">Módulo 1: Extracción de datos de diferentes fuentes: Archivos XML</h3>
    </div>
</div>


# Archivos XML

XML organiza la información como un árbol de elementos. Cada nodo puede tener una etiqueta, atributos, texto y nodos hijos. Esta jerarquía permite representar relaciones complejas, aunque requiere recorrer el árbol para llegar a los valores.

En los ejemplos se usan dos enfoques: `find`/`findall` para rutas conocidas y recorridos anidados cuando se necesita explorar la estructura. Al convertir XML a tabla hay que decidir qué nodo representa una fila y cómo tratar los elementos que faltan o se repiten.


El formato XML es común para el intercambio de datos estructurados. Python ofrece la librería estándar `xml.etree.ElementTree` para analizar y extraer información de archivos XML.

In [1]:
# Configuración común para ejecutar esta sección de forma independiente
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

dir_base = os.getcwd()
print(dir_base)
ruta = os.path.join(dir_base, 'Data')
print(ruta)

C:\Users\uie70742\OneDrive - ITESO\ITESO\Ingeniería y Ciencia de Datos\Laboratorio_de_Procesamiento_de_Datos_02026\Modulo I
C:\Users\uie70742\OneDrive - ITESO\ITESO\Ingeniería y Ciencia de Datos\Laboratorio_de_Procesamiento_de_Datos_02026\Modulo I\Data


In [2]:
import xml.etree.ElementTree as ET

In [3]:
xml_data = '''
<personas>
  <persona>
    <nombre>Ana</nombre>
    <edad>23</edad>
  </persona>
  <persona>
    <nombre>Luis</nombre>
    <edad>31</edad>
  </persona>
</personas>
'''

In [4]:
root = ET.fromstring(xml_data)
print('Personas extraídas del archivo XML:')
for persona in root.findall('persona'):
    nombre = persona.find('nombre').text
    edad = persona.find('edad').text
    print(f'Nombre: {nombre}, Edad: {edad}')

Personas extraídas del archivo XML:
Nombre: Ana, Edad: 23
Nombre: Luis, Edad: 31


### Anatomía de un documento XML

Un XML es un **árbol**: todo cuelga de un único elemento raíz. Cada **nodo** puede tener:

- una **etiqueta** (`tag`), p. ej. `<country>`;
- **atributos** (`attrib`), p. ej. `name="Singapore"`;
- **texto** (`text`), p. ej. `<rank>4</rank>`;
- **nodos hijos** anidados.

```mermaid
flowchart LR
    D["data (raíz)"] --> C1["country name='Liechtenstein'"]
    D --> C2["country name='Singapore'"]
    C1 --> R1["rank: 1"]
    C1 --> Y1["year: 2008"]
    C1 --> G1["gdppc: 141100"]
```

Con `xml.etree.ElementTree` recorremos ese árbol a mano; pero pandas ofrece además un atajo muy potente: `pd.read_xml`.

#### Métodos clave para navegar el árbol

| Método / atributo | Qué hace |
|---|---|
| `getroot()` | devuelve el nodo raíz |
| `nodo.tag` | nombre de la etiqueta |
| `nodo.attrib` | diccionario de atributos |
| `nodo.text` | texto contenido |
| `nodo.find('hijo')` | primer hijo que coincide |
| `nodo.findall('hijo')` | lista de hijos que coinciden |
| `nodo.iter('tag')` | recorre todos los descendientes con esa etiqueta, sin importar el nivel |

In [5]:
# Patrón muy común: recorrer el árbol y construir una lista de diccionarios
arbol = ET.parse(ruta + '/tabla_1.xml')
raiz = arbol.getroot()


In [6]:
raiz

<Element 'data' at 0x0000027301946C50>

In [7]:
#inspeccionar los datos del archivo xml
for nodo in raiz:
    print(nodo.attrib,nodo.text,nodo.tag)
    for sn in nodo:
        print(sn.attrib,sn.text,sn.tag)

{'name': 'Liechtenstein'} 
         country
{} 1 rank
{} 2008 year
{} 141100 gdppc
{'name': 'Singapore'} 
         country
{} 4 rank
{} 2011 year
{} 59900 gdppc
{'name': 'Panama'} 
         country
{} 68 rank
{} 2011 year
{} 13600 gdppc


In [8]:
#Extraer los datos de tabla_1.xml
d={}
for nodo in raiz:
    d[nodo.tag]=[]
for nodo in raiz:
    d[nodo.tag].append(nodo.attrib['name'])
for nodo in raiz:
    for sn in nodo:
        d[sn.tag]=[]
for nodo in raiz:
    for sn in nodo:
        d[sn.tag].append(sn.text)
d

{'country': ['Liechtenstein', 'Singapore', 'Panama'],
 'rank': ['1', '4', '68'],
 'year': ['2008', '2011', '2011'],
 'gdppc': ['141100', '59900', '13600']}

In [9]:
pd.DataFrame(d)

,country,rank,year,gdppc
0,Liechtenstein,1,2008,141100
1,Singapore,4,2011,59900
2,Panama,68,2011,13600


In [10]:
registros = []
for pais in raiz.findall('country'):
    registros.append({
        'pais': pais.attrib['name'],          # atributo del nodo
        'rank': int(pais.find('rank').text),  # texto de un hijo
        'year': int(pais.find('year').text),
        'gdppc': int(pais.find('gdppc').text)
    })

df_paises_xml = pd.DataFrame(registros)
df_paises_xml

,pais,rank,year,gdppc
0,Liechtenstein,1,2008,141100
1,Singapore,4,2011,59900
2,Panama,68,2011,13600


##### Ejemplo con tabla_2.xml

In [11]:
archivo_2=ET.parse(ruta+'/tabla_2.xml')
root=archivo_2.getroot()
for nodo in root:
    print(nodo.tag,nodo.attrib,nodo.text)

documents {'count': 'N'} 
        


In [12]:
for nodo in root:
    for subn in nodo:
        print(subn.tag,subn.attrib,subn.text)

document {'KEY': 'e95a9a6c790ecb95e46cf15bee517651', 'web': 'www.ubm/doc004.com'} Nevertheless, high and based on experience, fluctuating throughput levels contradict high reliability.

        
document {'KEY': 'bc360cfbafc39970587547215162f0db', 'web': 'www.ubm/doc006.com'} As long as the production is not conducted on the basis of concrete customer orders, the companies objectives

        
document {'KEY': '19e71144c50a8b9160b3f0955e906fce', 'web': 'www.ubm/doc005.com'} Many companies today, in different fields of operations and sizes, have access to a vast amount of data which was not available only a couple of years ago.

        
document {'KEY': '21d4af9021a174f61b884606c74d9e42', 'web': 'www.ubm/doc002.com'} A famous example of a successful prediction is by the German astronomer Johann Gottfried Galle

        
document {'KEY': '28a45eb2460899763d709ca00ddbb665', 'web': 'www.ubm/doc001.com'} A more recent example of the same kind is the prediction of the Higgs boson by Francoi

In [13]:
archivo=ET.parse(ruta+'/tabla_2.xml')
raiz=archivo.getroot()

In [14]:
L=[]
for n in raiz.findall('documents/document'):
    d={}
    d[n.tag]=n.text
    for k,v in n.attrib.items():
        d[k]=v
    L.append(d)
pd.DataFrame(L)

,document,KEY,web
0,"Nevertheless, high and based on experience, fl...",e95a9a6c790ecb95e46cf15bee517651,www.ubm/doc004.com
1,As long as the production is not conducted on ...,bc360cfbafc39970587547215162f0db,www.ubm/doc006.com
2,"Many companies today, in different fields of o...",19e71144c50a8b9160b3f0955e906fce,www.ubm/doc005.com
3,A famous example of a successful prediction is...,21d4af9021a174f61b884606c74d9e42,www.ubm/doc002.com
4,A more recent example of the same kind is the ...,28a45eb2460899763d709ca00ddbb665,www.ubm/doc001.com


#### Otro Ejemplo

In [15]:
archivo='IFC-Subscriptions-and-Voting-Power-of-Member-Count.xml'
file=ET.parse(os.path.join(ruta, archivo))
root=file.getroot()

for nodo in root:
    for snodo in nodo:
        print(snodo.tag,snodo.attrib,snodo.text)
        for ssnodo in snodo:
            print(ssnodo.tag,ssnodo.attrib,ssnodo.text)

row {'_id': 'row-y44j~at3a-b2ir', '_uuid': '00000000-0000-0000-0100-063657E78388', '_position': '0', '_address': 'https://finances.worldbank.org/resource/gsdw-avpz/row-y44j~at3a-b2ir'} 

member {} Afghanistan
amount_thousands_of_usd {} 1727
percent_of_total_amount {} 0.01
number_of_votes {} 8326
percent_of_total_votes {} 0.04
as_of_date {} 2021-08-06T00:00:00
row {'_id': 'row-cgqt~xxg3~xehh', '_uuid': '00000000-0000-0000-028B-9164C4785E6F', '_position': '0', '_address': 'https://finances.worldbank.org/resource/gsdw-avpz/row-cgqt~xxg3~xehh'} 

member {} Albania
amount_thousands_of_usd {} 9927
percent_of_total_amount {} 0.05
number_of_votes {} 16526
percent_of_total_votes {} 0.08
as_of_date {} 2021-08-06T00:00:00
row {'_id': 'row-q7xr_qk5p_hqh3', '_uuid': '00000000-0000-0000-E37B-CAECEC3A0C26', '_position': '0', '_address': 'https://finances.worldbank.org/resource/gsdw-avpz/row-q7xr_qk5p_hqh3'} 

member {} Algeria
amount_thousands_of_usd {} 51116
percent_of_total_amount {} 0.25
number_of

In [16]:
# Lista para almacenar los datos
datos = []

# Iterar sobre cada fila en el XML 
# root.iter("row")  suele ser una forma más directa de recorrer todos los elementos <row> descendientes.
for row in root.findall(".//row"):
    datos_fila = {
        '_id': row.attrib.get('_id', ''),
        '_uuid': row.attrib.get('_uuid', ''),
        '_position': row.attrib.get('_position', ''),
        '_address': row.attrib.get('_address', ''),
        'member': row.findtext("member", ''),
        'amount_thousands_of_usd': row.findtext("amount_thousands_of_usd", ''),
        'percent_of_total_amount': row.findtext("percent_of_total_amount", ''),
        'number_of_votes': row.findtext("number_of_votes", ''),
        'percent_of_total_votes': row.findtext("percent_of_total_votes", ''),
        'as_of_date': row.findtext("as_of_date", '')
    }
    datos.append(datos_fila)

# Crear DataFrame
df = pd.DataFrame(datos)
df

,_id,_uuid,_position,_address,member,amount_thousands_of_usd,percent_of_total_amount,number_of_votes,percent_of_total_votes,as_of_date
0,,,,,,,,,,
1,row-y44j~at3a-b2ir,00000000-0000-0000-0100-063657E78388,0,https://finances.worldbank.org/resource/gsdw-a...,Afghanistan,1727,0.01,8326,0.04,2021-08-06T00:00:00
2,row-cgqt~xxg3~xehh,00000000-0000-0000-028B-9164C4785E6F,0,https://finances.worldbank.org/resource/gsdw-a...,Albania,9927,0.05,16526,0.08,2021-08-06T00:00:00
3,row-q7xr_qk5p_hqh3,00000000-0000-0000-E37B-CAECEC3A0C26,0,https://finances.worldbank.org/resource/gsdw-a...,Algeria,51116,0.25,57715,0.26,2021-08-06T00:00:00
4,row-u2xu-enky.kggg,00000000-0000-0000-5E03-D3BA6F640E74,0,https://finances.worldbank.org/resource/gsdw-a...,Angola,11292,0.05,17891,0.08,2021-08-06T00:00:00
...,...,...,...,...,...,...,...,...,...,...
181,row-qrgm-r2gc-vmf5,00000000-0000-0000-6B23-1E7100931F81,0,https://finances.worldbank.org/resource/gsdw-a...,"Venezuela, Republica Bolivariana de",210347,1.01,216946,0.99,2021-08-06T00:00:00
182,row-v949_etve.z4iu,00000000-0000-0000-D24E-335270F95588,0,https://finances.worldbank.org/resource/gsdw-a...,Vietnam,3401,0.02,10000,0.05,2021-08-06T00:00:00
183,row-44vq_uwjv-izcc,00000000-0000-0000-2518-4C1118CD0F83,0,https://finances.worldbank.org/resource/gsdw-a...,"Yemen, Republic of",5452,0.03,12051,0.05,2021-08-06T00:00:00
184,row-pc34.25ft~32ca,00000000-0000-0000-A697-BD6CF2EAF622,0,https://finances.worldbank.org/resource/gsdw-a...,Zambia,9805,0.05,16404,0.07,2021-08-06T00:00:00


#### El atajo moderno: `pd.read_xml`

Cuando el XML tiene una estructura regular (un nodo repetido por fila), `pd.read_xml` construye el `DataFrame` en **una sola línea**, detectando etiquetas hijas y atributos automáticamente. Hace lo mismo que el bucle anterior, pero sin escribirlo. (Requiere el paquete `lxml`.)

In [17]:
# pd.read_xml hace en una línea lo que arriba hicimos a mano
pd.read_xml(ruta + '/tabla_1.xml')

,name,rank,year,gdppc
0,Liechtenstein,1,2008,141100
1,Singapore,4,2011,59900
2,Panama,68,2011,13600


#### Buscar en profundidad con `iter` y leer atributos

En árboles más profundos, `iter('tag')` encuentra **todos** los descendientes con esa etiqueta sin importar su nivel. En `tabla_2.xml` cada `<document>` guarda su texto en una sección `CDATA` y sus metadatos (clave, URL) en **atributos**.

In [18]:
# Extraer cada documento con su clave, URL y un fragmento del texto
arbol2 = ET.parse(ruta + '/tabla_2.xml')
raiz2 = arbol2.getroot()

docs = []
for doc in raiz2.iter('document'):
    texto = (doc.text or '').strip()
    docs.append({
        'key': doc.attrib.get('KEY'),
        'web': doc.attrib.get('web'),
        'fragmento': texto[:50] + '...'
    })

df_docs = pd.DataFrame(docs)
df_docs

,key,web,fragmento
0,e95a9a6c790ecb95e46cf15bee517651,www.ubm/doc004.com,"Nevertheless, high and based on experience, fl..."
1,bc360cfbafc39970587547215162f0db,www.ubm/doc006.com,As long as the production is not conducted on ...
2,19e71144c50a8b9160b3f0955e906fce,www.ubm/doc005.com,"Many companies today, in different fields of o..."
3,21d4af9021a174f61b884606c74d9e42,www.ubm/doc002.com,A famous example of a successful prediction is...
4,28a45eb2460899763d709ca00ddbb665,www.ubm/doc001.com,A more recent example of the same kind is the ...


In [19]:
# Los atributos también son datos: metadatos del autor (nodo raíz)
print('Atributos del autor :', raiz2.attrib)
print('Documentos encontrados:', len(list(raiz2.iter('document'))))

Atributos del autor : {'type': '001', 'language': 'EN', 'gender': 'h', 'feature': '00', 'web': '001_00.com'}
Documentos encontrados: 5


## Practica de Laboratorio: Extracción de Archivos XML

Utiliza el archivo `Data/plant_catalog.xml` para construir un pequeño catálogo de plantas en formato tabular (DataFrame). El XML contiene una colección de elementos `PLANT` y, dentro de cada uno, las etiquetas `COMMON`, `BOTANICAL`, `ZONE`, `LIGHT`, `PRICE` y `AVAILABILITY`. Extrae la información del XML, transformarla en un `DataFrame` y realiza consultas de interés, al final exporta el resultado a un archivo CSV.

#### Conteste las siguiente Preguntas

1. ¿Cuántas plantas contiene el catálogo?
2. ¿Cuántas plantas hay en cada zona?
3. ¿Qué tipo de iluminación aparece con mayor frecuencia?
4. ¿Qué nombre científico se repite más veces?


In [20]:
import xml.etree.ElementTree as ET

In [21]:
ruta

'C:\\Users\\uie70742\\OneDrive - ITESO\\ITESO\\Ingeniería y Ciencia de Datos\\Laboratorio_de_Procesamiento_de_Datos_02026\\Modulo I\\Data'

In [22]:
# 1. Lee el archivo utilizando `xml.etree.ElementTree`.
plantas = ET.parse(ruta + '/plant_catalog.xml')

In [23]:
# 2. Identifica el elemento raíz y cuenta cuántas plantas contiene.
raiz = plantas.getroot()
raiz

<Element 'CATALOG' at 0x0000027303A75B20>

In [24]:
# 3. Recorre cada elemento `PLANT` y extrae sus etiquetas.
for nodo in raiz:
    for snodo in nodo:
        print(snodo.tag,snodo.attrib,snodo.text)
        for ssnodo in snodo:
            print(ssnodo.tag,ssnodo.attrib,ssnodo.text)

COMMON {} Bloodroot
BOTANICAL {} Sanguinaria canadensis
ZONE {} 4
LIGHT {} Mostly Shady
PRICE {} $2.44
AVAILABILITY {} 031599
COMMON {} Columbine
BOTANICAL {} Aquilegia canadensis
ZONE {} 3
LIGHT {} Mostly Shady
PRICE {} $9.37
AVAILABILITY {} 030699
COMMON {} Marsh Marigold
BOTANICAL {} Caltha palustris
ZONE {} 4
LIGHT {} Mostly Sunny
PRICE {} $6.81
AVAILABILITY {} 051799
COMMON {} Cowslip
BOTANICAL {} Caltha palustris
ZONE {} 4
LIGHT {} Mostly Shady
PRICE {} $9.90
AVAILABILITY {} 030699
COMMON {} Dutchman's-Breeches
BOTANICAL {} Dicentra cucullaria
ZONE {} 3
LIGHT {} Mostly Shady
PRICE {} $6.44
AVAILABILITY {} 012099
COMMON {} Ginger, Wild
BOTANICAL {} Asarum canadense
ZONE {} 3
LIGHT {} Mostly Shady
PRICE {} $9.03
AVAILABILITY {} 041899
COMMON {} Hepatica
BOTANICAL {} Hepatica americana
ZONE {} 4
LIGHT {} Mostly Shady
PRICE {} $4.45
AVAILABILITY {} 012699
COMMON {} Liverleaf
BOTANICAL {} Hepatica americana
ZONE {} 4
LIGHT {} Mostly Shady
PRICE {} $3.99
AVAILABILITY {} 010299
COMMON {

In [25]:
# 4. Construye un `DataFrame` con las columnas: `nombre_comun`,`nombre_cientifico`,`zona`,`luz`,`precio`,`disponibilidad`
registros =[]
for planta in raiz.findall('PLANT'):
    registros.append({
        'nombre_comun': planta.findtext('COMMON'),
        'nombre_cientifico': planta.findtext('BOTANICAL'),
        'Zona': planta.findtext('ZONE'),
        'luz': planta.findtext('LIGHT'),
        'Precio': planta.findtext('PRICE'),
        'disponibilidad': planta.findtext('AVAILABILITY'),
    })
df_plantas = pd.DataFrame(registros)
df_plantas

,nombre_comun,nombre_cientifico,Zona,luz,Precio,disponibilidad
0,Bloodroot,Sanguinaria canadensis,4,Mostly Shady,$2.44,031599
1,Columbine,Aquilegia canadensis,3,Mostly Shady,$9.37,030699
2,Marsh Marigold,Caltha palustris,4,Mostly Sunny,$6.81,051799
3,Cowslip,Caltha palustris,4,Mostly Shady,$9.90,030699
4,Dutchman's-Breeches,Dicentra cucullaria,3,Mostly Shady,$6.44,012099
5,"Ginger, Wild",Asarum canadense,3,Mostly Shady,$9.03,041899
6,Hepatica,Hepatica americana,4,Mostly Shady,$4.45,012699
7,Liverleaf,Hepatica americana,4,Mostly Shady,$3.99,010299
8,Jack-In-The-Pulpit,Arisaema triphyllum,4,Mostly Shady,$3.23,020199
9,Mayapple,Podophyllum peltatum,3,Mostly Shady,$2.98,060599


In [26]:
df_plantas.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 36 entries, 0 to 35
Data columns (total 6 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   nombre_comun       36 non-null     object
 1   nombre_cientifico  36 non-null     object
 2   Zona               36 non-null     object
 3   luz                36 non-null     object
 4   Precio             36 non-null     object
 5   disponibilidad     36 non-null     object
dtypes: object(6)
memory usage: 1.8+ KB


In [27]:
# 5. Exporta el resultado a `Data/plant_catalog.csv`.
df_plantas.to_csv(ruta + '/plant_catalog.csv')

In [ ]:
# 6. Lee nuevamente el CSV con `pd.read_csv()` y verifica que conserve el mismo número de registros.
